# GovDoc-RAG: Benchmark A Index GPU Embedding & Chroma Export
This notebook embeds the 1,000 held-out PMIndia English articles for Benchmark A using `BAAI/bge-m3` on GPU (Kaggle/Colab).
It outputs:
1. `benchmark_A_embeddings.parquet`: Parquet table with `doc_id`, `text`, and L2-normalized 1024-dim embedding vectors.
2. `chroma_benchmark_A/`: Pre-indexed persistent ChromaDB collection ready for instant offline evaluation.
3. `chroma_benchmark_A.zip`: Downloadable archive.

In [ ]:
# 1. Environment & Dependencies
!nvidia-smi
!pip install torch transformers chromadb pyarrow pandas tqdm sentencepiece

In [ ]:
# 2. Clone repository (or upload data/eval/benchmark_A_index.jsonl)
import os
if not os.path.exists('govdoc-rag'):
    !git clone https://github.com/your-username/govdoc-rag.git
    os.chdir('govdoc-rag')
print('Working directory:', os.getcwd())

In [ ]:
# 3. Load Benchmark A Index Dataset
import json
from pathlib import Path

index_path = Path('data/eval/benchmark_A_index.jsonl')
assert index_path.exists(), f'Index file not found at {index_path}'

docs = []
with open(index_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            docs.append(json.loads(line))

print(f'Successfully loaded {len(docs)} documents for embedding.')
print('Sample doc_id:', docs[0]['doc_id'])
print('Sample text excerpt:', docs[0]['text'][:150])

In [ ]:
# 4. Load BAAI/bge-m3 Model on GPU
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

model_name = 'BAAI/bge-m3'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)
model.eval()
print('Model successfully loaded on GPU.')

In [ ]:
# 5. Batch Embedding Generation with L2 Normalization
from tqdm import tqdm
import numpy as np

batch_size = 16
all_embeddings = []
start_time = time.time()

with torch.no_grad():
    for i in tqdm(range(0, len(docs), batch_size), desc='Embedding'):
        batch_texts = [d['text'] for d in docs[i:i + batch_size]]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=8192,
            return_tensors='pt'
        ).to(device)
        
        outputs = model(**inputs)
        # Dense representation is CLS token embedding (dense_vecs)
        dense_vecs = outputs.last_hidden_state[:, 0]
        # L2-normalize per Rule 3 / requirements.md
        dense_vecs = F.normalize(dense_vecs, p=2, dim=1)
        all_embeddings.append(dense_vecs.cpu().float().numpy())

embeddings_arr = np.vstack(all_embeddings)
elapsed = time.time() - start_time

print(f'Embedded {len(docs)} documents in {elapsed:.2f}s ({len(docs) / elapsed:.1f} docs/sec).')
print(f'Embedding array shape: {embeddings_arr.shape}, dtype: {embeddings_arr.dtype}')
norms = np.linalg.norm(embeddings_arr, axis=1)
print(f'Norm verification: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}')

In [ ]:
# 6. Export to Parquet
import pandas as pd

df = pd.DataFrame({
    'doc_id': [d['doc_id'] for d in docs],
    'text': [d['text'] for d in docs],
    'embedding': [vec.tolist() for vec in embeddings_arr]
})

parquet_output = Path('data/eval/benchmark_A_embeddings.parquet')
parquet_output.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(parquet_output, index=False)
print(f'Exported {len(df)} embeddings to Parquet: {parquet_output} ({parquet_output.stat().st_size / 1e6:.2f} MB)')

In [ ]:
# 7. Export to ChromaDB Collection
import chromadb
import shutil

chroma_dir = Path('data/eval/chroma_benchmark_A')
if chroma_dir.exists():
    shutil.rmtree(chroma_dir)

client = chromadb.PersistentClient(path=str(chroma_dir))
# Must specify cosine space per Non-Negotiable Technical Rule 3
collection = client.get_or_create_collection(
    name='benchmark_A',
    metadata={'hnsw:space': 'cosine'}
)

# Ingest in chunks of 200 into ChromaDB
chunk_size = 200
for i in range(0, len(docs), chunk_size):
    batch_docs = docs[i:i + chunk_size]
    batch_vecs = embeddings_arr[i:i + chunk_size].tolist()
    collection.add(
        ids=[f'doc_{j}' for j in range(i, i + len(batch_docs))],
        embeddings=batch_vecs,
        documents=[d['text'] for d in batch_docs],
        metadatas=[{'doc_id': d['doc_id']} for d in batch_docs]
    )

print(f'Chroma collection count: {collection.count()}')

# Archive for offline transfer
shutil.make_archive('chroma_benchmark_A', 'zip', str(chroma_dir))
print('Chroma DB zipped to chroma_benchmark_A.zip ready for download.')